# 05 · Previsões com o modelo vigente

**Objetivo.** Gerar previsões de inadimplência para clientes que o modelo nunca viu, usando o modelo vigente, e provar que elas são reproduzíveis.

**Entradas.**
- `models/modelo_vigente.json` e o pacote `models/versoes/<model_id>/modelo.joblib`, que contém o pipeline (pré-processador + estimador) e os atributos eleitos.
- `data/raw/inferencia/novos_clientes.csv`: 3.000 clientes separados da fonte na ingestão, **sem rótulo**. Eles não estão em nenhum lote de treino, validação ou teste.

**Saídas.**
- `reports/previsoes/previsoes_<cycle_id>.csv`: ID, probabilidade, classe prevista, `model_id`, `cycle_id` e `data_version`.
- `reports/evidencias/<cycle_id>/05_previsoes.json` e o run filho `previsoes` no MLflow.

**Metodologia.**
1. Carregar o pacote e conferir o SHA-256 e os atributos contra o manifesto.
2. Aplicar aos registros novos as mesmas transformações de linha do ETL (recodificação + atributos derivados) via `pp.prepare_inference`.
3. Prever usando o pré-processador já ajustado no treino do modelo. Nada é reajustado.
4. Reproduzir: recarregar o modelo do disco e também do MLflow e confirmar as mesmas classes e probabilidades (até `1e-12`).

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config" / "config.yaml").is_file())
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

import joblib
import mlflow
import mlflow.sklearn
import numpy as np

from src import preprocessamento as pp, promocao
from src.utils import configure_mlflow, cycle_run, load_config, load_cycle, resolve_path, sha256_file, write_json

pd.set_option("display.max_columns", 50)
cfg = load_config(ROOT)
TARGET = cfg["data"]["target"]

In [2]:
ctx = load_cycle(cfg, ROOT)
configure_mlflow(cfg, ROOT)
active = promocao.load_active_model(cfg, ROOT)
if active is None:
    raise RuntimeError("Nenhum modelo vigente. Execute o notebook 04 primeiro.")
bundle, manifest = active

print("modelo vigente :", manifest["model_id"])
print("algoritmo      :", manifest["algorithm"], "| promovido no ciclo", manifest["cycle_id"])
print("atributos      :", bundle["features"])
print(f"{manifest['primary_metric']} validação: {manifest['metric_value']:.4f}")

modelo vigente : ciclo_20261116T223751Z_a2d3d2d2__random_forest
algoritmo      : random_forest | promovido no ciclo ciclo_20261116T223751Z_a2d3d2d2
atributos      : ['LIMIT_BAL', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_AMT1', 'UTILIZACAO_MEDIA', 'MESES_EM_ATRASO', 'ATRASO_MAXIMO']
f1 validação: 0.5439


## 1. Registros novos

In [3]:
raw = pd.read_csv(resolve_path(cfg, "inference_file", ROOT))
new_clients, prep_log = pp.prepare_inference(raw, cfg)
assert not set(new_clients["ID"]) & set(pp.load_processed(cfg, ctx, ROOT)["ID"]), "clientes de inferência nunca vistos no treino"
print(raw.shape, "->", new_clients.shape, prep_log)

(3000, 24) -> (3000, 30) {'recodificados_EDUCATION': 37, 'recodificados_MARRIAGE': 5, 'linhas_sem_previsao': 0}


## 2. Previsões

In [4]:
X_new = new_clients[bundle["features"]]
proba = bundle["pipeline"].predict_proba(X_new)[:, 1]
pred = bundle["pipeline"].predict(X_new)

predictions = pd.DataFrame({"ID": new_clients["ID"], "probabilidade_inadimplencia": proba.round(6), "previsao": pred,
                            "model_id": manifest["model_id"], "cycle_id": ctx.cycle_id, "data_version": ctx.data_version})
out = resolve_path(cfg, "predictions_dir", ROOT) / f"previsoes_{ctx.cycle_id}.csv"
out.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(out, index=False)

display(predictions.head())
print(f"{len(predictions)} previsões | taxa prevista de inadimplência: {pred.mean():.2%} -> {out.relative_to(ROOT)}")

,ID,probabilidade_inadimplencia,previsao,model_id,cycle_id,data_version
0,11,0.405008,0,ciclo_20261116T223751Z_a2d3d2d2__random_forest,ciclo_20261116T223819Z_2db70707,dv_2db70707140b1359
1,24,0.108142,0,ciclo_20261116T223751Z_a2d3d2d2__random_forest,ciclo_20261116T223819Z_2db70707,dv_2db70707140b1359
2,28,0.418605,0,ciclo_20261116T223751Z_a2d3d2d2__random_forest,ciclo_20261116T223819Z_2db70707,dv_2db70707140b1359
3,34,0.199488,0,ciclo_20261116T223751Z_a2d3d2d2__random_forest,ciclo_20261116T223819Z_2db70707,dv_2db70707140b1359
4,53,0.163933,0,ciclo_20261116T223751Z_a2d3d2d2__random_forest,ciclo_20261116T223819Z_2db70707,dv_2db70707140b1359


3000 previsões | taxa prevista de inadimplência: 27.70% -> reports/previsoes/previsoes_ciclo_20261116T223819Z_2db70707.csv


## 3. Reprodutibilidade

O modelo é recarregado de duas fontes independentes e as previsões são comparadas com as que acabaram de ser gravadas:
- o arquivo `joblib` versionado em `models/versoes/`;
- o artefato do run do vencedor no MLflow (`model_uri` do manifesto).

In [5]:
reloaded = joblib.load(ROOT / manifest["artifact_path"])
disk_model = reloaded["pipeline"]
mlflow_model = mlflow.sklearn.load_model(manifest["model_uri"])
saved = pd.read_csv(out)

TOL = 1e-12
checks = {
    "joblib: mesmas classes": np.array_equal(disk_model.predict(new_clients[reloaded["features"]]), pred),
    "joblib: mesmas probabilidades": np.allclose(disk_model.predict_proba(new_clients[reloaded["features"]])[:, 1], proba, rtol=0, atol=TOL),
    "MLflow: mesmas classes": np.array_equal(mlflow_model.predict(X_new), pred),
    "MLflow: mesmas probabilidades": np.allclose(mlflow_model.predict_proba(X_new)[:, 1], proba, rtol=0, atol=TOL),
    "CSV gravado: mesmas classes": np.array_equal(saved["previsao"], pred),
}
display(pd.Series(checks, name="ok").to_frame())
reproducible = all(checks.values())
assert reproducible, "previsões precisam ser reproduzíveis a partir do modelo persistido"

,ok
joblib: mesmas classes,True
joblib: mesmas probabilidades,True
MLflow: mesmas classes,True
MLflow: mesmas probabilidades,True
CSV gravado: mesmas classes,True


A tolerância de `1e-12` nas probabilidades não é folga estatística. A Random Forest soma as probabilidades das árvores em paralelo (`n_jobs=-1`), e a ordem dessa soma entre threads muda os últimos bits do ponto flutuante (diferenças da ordem de `1e-16`). As classes previstas são idênticas.

## 4. Registro

In [6]:
summary = {**ctx.tags, "model_id": manifest["model_id"], "model_cycle_id": manifest["cycle_id"],
           "n_previsoes": len(predictions), "taxa_prevista_inadimplencia": float(pred.mean()),
           "probabilidade_media": float(proba.mean()), "arquivo": out.relative_to(ROOT).as_posix(),
           "sha256": sha256_file(out), "reproducible": reproducible, **prep_log}
write_json(ctx.evidence_dir(cfg, ROOT) / "05_previsoes.json", summary)

with cycle_run(ctx, cfg, stage="previsoes", root=ROOT):
    mlflow.set_tags({"active_model_id": manifest["model_id"]})
    mlflow.log_params({"model_id": manifest["model_id"], "n_previsoes": len(predictions)})
    mlflow.log_metrics({"taxa_prevista_inadimplencia": summary["taxa_prevista_inadimplencia"],
                        "probabilidade_media": summary["probabilidade_media"], "reproducible": int(reproducible)})
    mlflow.log_artifact(str(out))
display(pd.Series(summary, name="valor").to_frame())

,valor
cycle_id,ciclo_20261116T223819Z_2db70707
data_version,dv_2db70707140b1359
model_id,ciclo_20261116T223751Z_a2d3d2d2__random_forest
model_cycle_id,ciclo_20261116T223751Z_a2d3d2d2
n_previsoes,3000
taxa_prevista_inadimplencia,0.277
probabilidade_media,0.426544
arquivo,reports/previsoes/previsoes_ciclo_20261116T223...
sha256,55ba99ac3d02879e8bda8233f07ed2f3b0a29e38f622e4...
reproducible,True


## Conclusões

- As previsões usam o modelo vigente junto com o pré-processador e a lista de atributos com que ele foi treinado. Os três viajam no mesmo pacote, conferido contra o manifesto por SHA-256.
- Os registros de inferência nunca participaram do treino, da validação ou do teste.
- As previsões foram reproduzidas a partir do `joblib` versionado e do artefato no MLflow: classes idênticas e probabilidades iguais até `1e-12`.
- Cada arquivo de previsões leva `model_id`, `cycle_id` e `data_version`. Assim, toda previsão é rastreável até o modelo e os dados que a geraram.